# Knowledge Distillation — Distilling the Knowledge in a Neural Network (2015)

_Papers · Efficiency & Compression_

**Train a small model to copy a big model's softened class probabilities, not just the hard labels.**

---

This notebook is a practice scaffold for the **Knowledge Distillation — Distilling the Knowledge in a Neural Network (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 0. Sanity-check the lesson's worked example: softened softmax at T=1 vs T=4. ---
z = torch.tensor([2.0, 1.0, 0.1, -1.0])
for T in (1.0, 4.0):
    q = F.softmax(z / T, dim=0)
    print(f"T={T}: ", [round(v, 4) for v in q.tolist()])
# T=1.0:  [0.6381, 0.2347, 0.0954, 0.0318]
# T=4.0:  [0.3481, 0.2711, 0.2165, 0.1644]


# --- 1. A small, noisy toy classification problem (where a tiny student under-fits). ---
g = torch.Generator().manual_seed(1)
K, n = 10, 30                       # 10 classes, 30-dim inputs
centers = torch.randn(K, n, generator=g) * 1.6
def make(N):
    y = torch.randint(0, K, (N,), generator=g)
    X = centers[y] + torch.randn(N, n, generator=g) * 3.2   # heavy overlap -> hard
    return X, y
Xtr, ytr = make(300)                # SMALL train set -> soft targets help most
Xte, yte = make(2000)


class Net(nn.Module):
    def __init__(self, h, layers=2):
        super().__init__()
        mods = [nn.Linear(n, h), nn.ReLU()]
        for _ in range(layers - 1):
            mods += [nn.Linear(h, h), nn.ReLU()]
        mods += [nn.Linear(h, K)]
        self.net = nn.Sequential(*mods)
    def forward(self, x):
        return self.net(x)


def acc(net, X, y):
    net.eval()
    with torch.no_grad():
        return (net(X).argmax(1) == y).float().mean().item()


def train_hard(net, epochs, lr=0.01, wd=0.0):
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss(); net.train()
    for _ in range(epochs):
        opt.zero_grad(); ce(net(Xtr), ytr).backward(); opt.step()
    return net


# --- 2. THE NOVEL PART: the combined distillation loss. ---
def train_distill(net, teacher, T, alpha=0.7, epochs=600, lr=0.01):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    ce  = nn.CrossEntropyLoss()
    teacher.eval()
    with torch.no_grad():
        soft_t = F.softmax(teacher(Xtr) / T, dim=1)        # Eqn. 1: softened TEACHER targets (frozen)
    net.train()
    for _ in range(epochs):
        opt.zero_grad()
        z_s = net(Xtr)                                     # student logits
        log_soft_s = F.log_softmax(z_s / T, dim=1)         # Eqn. 1: softened STUDENT, in log-space
        kd   = F.kl_div(log_soft_s, soft_t, reduction="batchmean") * (T * T)   # the T^2 rule
        hard = ce(z_s, ytr)                                # hard-label cross-entropy, at T=1
        loss = alpha * kd + (1 - alpha) * hard             # weighted average of the two objectives
        loss.backward(); opt.step()
    return net


# --- 3. Train a big teacher, then two matched small students. ---
torch.manual_seed(0); teacher = train_hard(Net(512, layers=3), epochs=2000, wd=1e-4)
print(f"Teacher (h=512, 3L)        test acc = {acc(teacher, Xte, yte):.4f}")

torch.manual_seed(0); s_hard = train_hard(Net(8, layers=2), epochs=600)
print(f"Student hard-only (h=8)    test acc = {acc(s_hard, Xte, yte):.4f}")

torch.manual_seed(0); s_kd = train_distill(Net(8, layers=2), teacher, T=4.0)
print(f"Student +KD (T=4, h=8)     test acc = {acc(s_kd, Xte, yte):.4f}")

# --- 4. Ablation: sweep the temperature. ---
print("Ablation over temperature T (student h=8, alpha=0.7):")
for T in (1.0, 2.0, 4.0, 8.0, 16.0):
    torch.manual_seed(0); s = train_distill(Net(8, layers=2), teacher, T=T)
    print(f"  T={T:<5} test acc = {acc(s, Xte, yte):.4f}")

# Our small run (CPU, seeds fixed), NOT the paper's reported numbers:
#   Teacher 0.7680 | Student hard-only 0.5970 | Student +KD (T=4) 0.6570
#   T-ablation: T=1 0.6070 | T=2 0.5995 | T=4 0.6570 | T=8 0.7100 | T=16 0.6755
# Distillation beats hard labels; accuracy traces an inverted-U in T (peaks near T=8).

## Visualize the data & results

_Does a small student trained with distillation beat the same student on hard labels alone, and how does test accuracy vary with the temperature T?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

g = torch.Generator().manual_seed(1)
K, n = 10, 30
centers = torch.randn(K, n, generator=g) * 1.6
def make(N):
    y = torch.randint(0, K, (N,), generator=g)
    X = centers[y] + torch.randn(N, n, generator=g) * 3.2
    return X, y
Xtr, ytr = make(300); Xte, yte = make(2000)

class Net(nn.Module):
    def __init__(self, h, layers=2):
        super().__init__()
        m = [nn.Linear(n, h), nn.ReLU()]
        for _ in range(layers - 1): m += [nn.Linear(h, h), nn.ReLU()]
        m += [nn.Linear(h, K)]; self.net = nn.Sequential(*m)
    def forward(self, x): return self.net(x)

def acc(net, X, y):
    net.eval()
    with torch.no_grad(): return (net(X).argmax(1) == y).float().mean().item()

def train_hard(net, epochs, wd=0.0):
    opt = torch.optim.Adam(net.parameters(), lr=0.01, weight_decay=wd)
    ce = nn.CrossEntropyLoss(); net.train()
    for _ in range(epochs):
        opt.zero_grad(); ce(net(Xtr), ytr).backward(); opt.step()
    return net

def train_distill(net, teacher, T, alpha=0.7, epochs=600):
    opt = torch.optim.Adam(net.parameters(), lr=0.01); ce = nn.CrossEntropyLoss()
    teacher.eval()
    with torch.no_grad(): soft_t = F.softmax(teacher(Xtr) / T, dim=1)
    net.train()
    for _ in range(epochs):
        opt.zero_grad(); z = net(Xtr)
        kd = F.kl_div(F.log_softmax(z / T, dim=1), soft_t, reduction="batchmean") * (T * T)
        loss = alpha * kd + (1 - alpha) * ce(z, ytr)
        loss.backward(); opt.step()
    return net

torch.manual_seed(0); teacher = train_hard(Net(512, 3), 2000, wd=1e-4)
torch.manual_seed(0); hard = acc(train_hard(Net(8, 2), 600), Xte, yte)
rows = [["hard-only", round(hard, 4)]]
for T in (1.0, 2.0, 4.0, 8.0, 16.0):
    torch.manual_seed(0)
    rows.append([f"KD T={int(T)}", round(acc(train_distill(Net(8, 2), teacher, T), Xte, yte), 4)])
rows.append(["teacher", round(acc(teacher, Xte, yte), 4)])
for r in rows: print(r)
# [['hard-only', 0.597], ['KD T=1', 0.607], ['KD T=2', 0.5995], ['KD T=4', 0.657],
#  ['KD T=8', 0.71], ['KD T=16', 0.6755], ['teacher', 0.768]]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline effect. You train one small student on hard labels alone and an identical
            student with the distillation loss (soft teacher targets at $T=4$ plus hard labels). On a small,
            noisy training set, which reaches higher test accuracy, and what does that show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Keep everything identical &mdash; same student architecture, optimizer, seed, data &mdash; and change only whether the soft-target term is present. — _An honest comparison isolates the one variable: the soft targets._
- Train both, then evaluate at $T=1$. The distilled student lands higher (in our run, 0.6570 vs 0.5970). — _The soft targets act as a richer, lower-variance training signal than one-hot labels, regularizing the small student._
- Conclude that the teacher's softened probabilities &mdash; not extra parameters &mdash; closed part of the gap to the big teacher. — _Both students have identical capacity; only the training signal differs._

**Answer:** The distilled student wins. In our small run the hard-labels-only student reached
                 about 0.597 test accuracy, while the same student trained with soft targets at $T=4$
                 reached about 0.657 &mdash; closing much of the gap toward the teacher's 0.768.
                 Since the only difference is the soft-target term, this isolates the teacher's softened
                 probabilities as the cause. It is a regularization / information effect, not a
                 capacity one. (Our small run, not the paper's number.)

</details>

**Problem 2.** The temperature ablation. Holding the student, teacher, and weight $\alpha$ fixed, you
            sweep the distillation temperature $T \in \{1, 2, 4, 8, 16\}$. What shape do you expect the test
            accuracy to trace as $T$ grows, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- At $T=1$ the soft targets are nearly one-hot (the teacher is confident), so they carry little extra signal beyond the hard labels. — _Almost no "dark knowledge" is exposed; distillation ~ hard-label training._
- At moderate $T$ (say 4&ndash;8) the wrong-class probabilities become readable, giving the student the most useful structure. — _This is the sweet spot the paper relies on &mdash; informative but not washed out._
- At very large $T$ every class approaches equal probability, so the targets carry almost no class information and accuracy falls again. — _Over-softening destroys the signal; the inverted-U turns back down._

**Answer:** An inverted U: accuracy rises from $T=1$, peaks at a moderate temperature, then
                 falls as $T$ gets too large. In our run: $T{=}1 \to 0.607$, $T{=}2 \to 0.600$,
                 $T{=}4 \to 0.657$, $T{=}8 \to 0.710$ (best), $T{=}16 \to 0.676$. Too cold and the targets are
                 nearly one-hot; too hot and they are nearly uniform; in between they expose the most useful
                 dark knowledge. (Our small run, not the paper's number.)

</details>

**Problem 3.** Your worked example had logits $z = [2.0, 1.0, 0.1, -1.0]$, giving $T{=}1$ probabilities
            $[0.638, 0.235, 0.095, 0.032]$. Without softening, the smallest class is $0.032$. Why does the
            teacher's $0.032$ matter to the student, and how does raising $T$ help convey it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that $0.032$ is small but not zero &mdash; it says the teacher finds that class slightly plausible, more so than a class it would rate near $0$. — _These relative wrong-class odds are the "dark knowledge" &mdash; they encode the teacher's learned similarities._
- At $T=1$ a hard label would hand the student a flat $0$ for that class, discarding the nuance. Raise $T$ to 4: the class rises to $0.164$. — _Softening amplifies the small probabilities so the student's loss actually feels them._
- Train the student to match the softened vector, then test at $T=1$. — _The student absorbs the similarity structure during training, then reverts to ordinary probabilities for deployment._

**Answer:** The $0.032$ tells the student that the teacher considers that class a near-miss, not an
                 impossibility &mdash; structure a one-hot label throws away. At $T=1$ that signal is tiny and
                 easily ignored; raising $T$ to $4$ lifts it to $0.164$ (a $5\times$ jump), so the soft-target
                 loss actually transmits it. The student learns the teacher's class-similarity structure, then
                 switches back to $T=1$ at test time.

</details>